# When the Eval Schema Doesn't Match the Data

The eval schema is the single source of truth: the scorer trusts it completely.
So a schema that doesn't match the actual gold / extracted JSON does **not**
crash the eval -- it silently mis-scores. A perfect extraction can come out
with F1 = 0 and you'd blame the extractor.

This notebook walks through every way a schema can disagree with the filled-in
JSON, and what each one looks like in the results:

| # | Mismatch | What happens |
|---|----------|--------------|
| 1 | Data has a field the schema doesn't | correct value punished as **hallucination** |
| 2 | Field is at a different path than the schema says | same as 1 -- path is part of a field's identity |
| 3 | Schema has a field the data never has | harmless (warning only) |
| 4 | Extracted alone is missing / has extra fields | **not** a schema problem -- this is what the eval measures |
| 5 | Wrong declared leaf type | identical values score 0 with `reason=type_error` |
| 6 | Wrong declared container type | forgiving match, but partial credit is lost |
| 7 | Wrong array alignment key | perfect extraction scores F1 = 0 |
| 8 | Invalid `x-eval-*` values | fails fast at parse time, before any scoring |

The safety net for 1-2 is `validate_gold()` -- run it before every eval.

In [1]:
import logging

from struct_extract_eval import (
    GoldValidationError,
    annotate_xeval,
    evaluate,
    parse_eval_schema,
    validate_gold,
)

# Surface the library's warnings -- several mismatches below only warn.
logging.basicConfig(format="WARNING  %(message)s", level=logging.WARNING, force=True)


def show(schema, gold, extracted):
    """Score one gold-extracted pair and print per-field results + metrics."""
    annotate_xeval(schema)  # fill in default comparators (idempotent)
    run = evaluate([gold], [extracted], schema=schema)
    record = run.records[0]
    print(f"{'Field':<24} {'Status':<14} {'Score':>5}  Gold / Extracted")
    print("-" * 78)
    for r in record.field_results:
        line = (f"{r.path:<24} {r.status:<14} {r.score:>5.1f}  "
                f"{r.gold_value!r} / {r.extracted_value!r}")
        if r.reason:
            line += f"   ({r.reason})"
        print(line)
    print("-" * 78)
    print(f"precision={record.precision:.2f}  "
          f"recall={record.recall:.2f}  f1={record.f1:.2f}")

## Case 1: Data has a field the schema doesn't

The extractor got `temperature` exactly right -- but the schema doesn't know
the field exists. The gold copy is ignored, and the extracted copy is punished
as invented data.

This is the worst failure mode: hallucination drags **precision** down harder
than omission would, so a missing schema field makes your best extractor look
like a fabricator.

In [2]:
schema = {"type": "object", "properties": {
    "name": {"type": "string"},
    # "temperature" forgotten!
}}

gold      = {"name": "PbS", "temperature": 150}
extracted = {"name": "PbS", "temperature": 150}   # perfect extraction

show(schema, gold, extracted)

Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
name                     match            1.0  'PbS' / 'PbS'
temperature              hallucination    0.0  None / 150
------------------------------------------------------------------------------
precision=0.50  recall=1.00  f1=0.67


`validate_gold()` exists exactly for this. It checks that every gold field is
in the schema and raises before any scoring happens:

In [3]:
try:
    validate_gold([gold], schema)
except GoldValidationError as e:
    print(f"GoldValidationError: {e}")

GoldValidationError: Record 0: field 'temperature' is in gold but not in schema. All gold fields must be defined in the eval schema. If this field should not be scored, add it to the schema with x-eval-skip: true.


## Case 2: Field path doesn't match the schema

The schema nests `temperature` under `conditions`, but the data has it at the
top level. The **value** is right and the schema **does** have a temperature
field -- but path is part of a field's identity, so this behaves exactly like
Case 1: the schema's `conditions.temperature` is never scored (absent on both
sides), and the top-level `temperature` is a hallucination.

In [4]:
schema = {"type": "object", "properties": {
    "name": {"type": "string"},
    "conditions": {"type": "object", "properties": {
        "temperature": {"type": "number"},
    }},
}}

gold      = {"name": "PbS", "temperature": 150}   # flat, not nested
extracted = {"name": "PbS", "temperature": 150}   # perfect extraction

show(schema, gold, extracted)

Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
name                     match            1.0  'PbS' / 'PbS'
temperature              hallucination    0.0  None / 150
------------------------------------------------------------------------------
precision=0.50  recall=1.00  f1=0.67


The reverse (schema flat, data nested) fails the same way -- the whole
unexpected `conditions` dict becomes one hallucination:

In [5]:
schema = {"type": "object", "properties": {
    "name": {"type": "string"},
    "temperature": {"type": "number"},   # schema expects it flat
}}

gold      = {"name": "PbS", "conditions": {"temperature": 150}}
extracted = {"name": "PbS", "conditions": {"temperature": 150}}

show(schema, gold, extracted)

Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
name                     match            1.0  'PbS' / 'PbS'
conditions               hallucination    0.0  None / {'temperature': 150}
------------------------------------------------------------------------------
precision=0.50  recall=1.00  f1=0.67


Again, `validate_gold()` catches both directions, because in both cases the
gold has a key the schema doesn't define at that path:

In [6]:
try:
    validate_gold([gold], schema)
except GoldValidationError as e:
    print(f"GoldValidationError: {e}")

WARNING  Record 0: field 'temperature' is in schema but missing from gold. It will not be scored for this record.


GoldValidationError: Record 0: field 'conditions' is in gold but not in schema. All gold fields must be defined in the eval schema. If this field should not be scored, add it to the schema with x-eval-skip: true.


## Case 3: Schema has a field the data never has

Harmless. A field that is absent from **both** gold and extracted is simply
not scored -- it doesn't count toward precision, recall, or `total_fields`.
`validate_gold()` warns (so you notice dead schema fields) but doesn't raise.

In [7]:
schema = {"type": "object", "properties": {
    "name": {"type": "string"},
    "temperature": {"type": "number"},
    "pressure": {"type": "number"},   # never appears in the data
}}

gold      = {"name": "PbS", "temperature": 150}
extracted = {"name": "PbS", "temperature": 150}

annotate_xeval(schema)
validate_gold([gold], schema)   # warns about 'pressure', does not raise
show(schema, gold, extracted)

WARNING  Record 0: field 'pressure' is in schema but missing from gold. It will not be scored for this record.


Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
name                     match            1.0  'PbS' / 'PbS'
temperature              match            1.0  150 / 150
------------------------------------------------------------------------------
precision=1.00  recall=1.00  f1=1.00


## Case 4: For contrast -- the extractor (not the schema) is wrong

Missing and extra fields in the **extracted side only** are not schema
problems. They are exactly what the eval is designed to measure: omission
(hurts recall) and hallucination (hurts precision).

Note the symptom looks identical to Cases 1-2 -- a hallucination in the
results. The way to tell them apart: in a schema problem, **gold also
disagrees with the schema**, and `validate_gold()` raises. Here it passes.

In [8]:
schema = {"type": "object", "properties": {
    "name": {"type": "string"},
    "temperature": {"type": "number"},
    "thickness": {"type": "number"},
}}

gold      = {"name": "PbS", "temperature": 150, "thickness": 50.0}
extracted = {"name": "PbS", "purity": 0.999}
# extractor dropped temperature + thickness, invented purity

annotate_xeval(schema)
validate_gold([gold], schema)   # passes: gold matches the schema
show(schema, gold, extracted)

Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
name                     match            1.0  'PbS' / 'PbS'
temperature              omission         0.0  150 / None
thickness                omission         0.0  50.0 / None
purity                   hallucination    0.0  None / 0.999
------------------------------------------------------------------------------
precision=0.50  recall=0.33  f1=0.40


## Case 5: Wrong declared leaf type

`"type": "number"` assigns the `numeric` comparator by default. If the real
values are strings with units, `float("150 C")` fails and **identical** values
score 0 with `reason=type_error`.

Fix: declare the field as `string`, or add a transform that strips units, or
set an explicit `x-eval-compare` that handles the real shape.

In [9]:
schema = {"type": "object", "properties": {
    "temp": {"type": "number"},    # but the data carries units as strings
}}

gold      = {"temp": "150 C"}
extracted = {"temp": "150 C"}     # identical!

show(schema, gold, extracted)

Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
temp                     mismatch         0.0  '150 C' / '150 C'   (type_error)
------------------------------------------------------------------------------
precision=0.00  recall=0.00  f1=0.00


Clean numeric strings are forgiven -- the `numeric` comparator coerces with
`float()`, so `"150"` vs `150.0` still matches:

In [10]:
show({"type": "object", "properties": {"temp": {"type": "number"}}},
     {"temp": "150"}, {"temp": 150.0})

Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
temp                     match            1.0  '150' / 150.0
------------------------------------------------------------------------------
precision=1.00  recall=1.00  f1=1.00


## Case 6: Wrong declared container type

For objects and arrays, `json_type` is treated as a **hint, not a constraint**
(the field may be polymorphic). If the schema says `object` but both sides are
actually equal arrays, it's still a match -- a faithful extractor is never
punished for the schema's wrong guess.

The cost: element-level scoring is gone. The container is compared as one
opaque value, so one wrong element zeroes the whole thing -- no partial credit.

In [11]:
schema = {"type": "object", "properties": {
    "authors": {"type": "object", "properties": {   # wrong: data is an array
        "name": {"type": "string"},
    }},
}}

print("equal arrays under an 'object' schema:")
show(schema, {"authors": ["Alice", "Bob"]}, {"authors": ["Alice", "Bob"]})

print()
print("one element wrong -- whole container scores 0, no partial credit:")
show(schema, {"authors": ["Alice", "Bob"]}, {"authors": ["Alice", "Bobby"]})

equal arrays under an 'object' schema:
Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
authors                  match            1.0  ['Alice', 'Bob'] / ['Alice', 'Bob']
------------------------------------------------------------------------------
precision=1.00  recall=1.00  f1=1.00

one element wrong -- whole container scores 0, no partial credit:
Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
authors                  mismatch         0.0  ['Alice', 'Bob'] / ['Alice', 'Bobby']   (type mismatch: gold list, extracted list)
------------------------------------------------------------------------------
precision=0.00  recall=0.00  f1=0.00


## Case 7: Wrong array alignment key

The schema says match `steps` elements by `"id"` -- but the elements only have
`"name"` and `"time"`. Parsing only **warns** and proceeds. No gold element
can match any extracted element, so a **perfect extraction scores F1 = 0**:
every gold field becomes an omission, every extracted field a hallucination.

If an array scores all-omission + all-hallucination across every record,
suspect the alignment key before the extractor.

In [12]:
schema = {"type": "object", "properties": {
    "steps": {
        "type": "array",
        "x-eval-align": {"match_by": "key_field", "key": "id"},   # no such field
        "items": {"type": "object", "properties": {
            "name": {"type": "string"},
            "time": {"type": "number"},
        }},
    },
}}

gold      = {"steps": [{"name": "anneal", "time": 30}, {"name": "etch", "time": 5}]}
extracted = {"steps": [{"name": "etch", "time": 5}, {"name": "anneal", "time": 30}]}
# same steps, different order -- a perfect extraction

show(schema, gold, extracted)

WARNING  Path 'steps': x-eval-align key 'id' not found in items properties ['name', 'time']. Key-field matching will attempt to match on data anyway.


Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
steps[0].name            omission         0.0  'anneal' / None
steps[0].time            omission         0.0  30 / None
steps[1].name            omission         0.0  'etch' / None
steps[1].time            omission         0.0  5 / None
steps[-1].name           hallucination    0.0  None / 'etch'
steps[-1].time           hallucination    0.0  None / 5
steps[-2].name           hallucination    0.0  None / 'anneal'
steps[-2].time           hallucination    0.0  None / 30
------------------------------------------------------------------------------
precision=0.00  recall=0.00  f1=0.00


With the right key (`"name"`), the same pair scores perfectly:

In [13]:
schema["properties"]["steps"]["x-eval-align"] = {"match_by": "key_field", "key": "name"}
show(schema, gold, extracted)

Field                    Status         Score  Gold / Extracted
------------------------------------------------------------------------------
steps[0].name            match            1.0  'anneal' / 'anneal'
steps[0].time            match            1.0  30 / 30
steps[1].name            match            1.0  'etch' / 'etch'
steps[1].time            match            1.0  5 / 5
------------------------------------------------------------------------------
precision=1.00  recall=1.00  f1=1.00


## Case 8: Invalid x-eval config fails fast

Unknown comparator names and unknown alignment strategies are schema
**errors**, not data mismatches -- they raise `SchemaError` at parse time,
before any record is scored. These can never silently distort results.

In [14]:
from struct_extract_eval.core.schema.tree import SchemaError

bad_comparator = {"type": "object", "properties": {
    "name": {"type": "string", "x-eval-compare": "smart_match"},
}}
try:
    parse_eval_schema(annotate_xeval(bad_comparator))
except SchemaError as e:
    print(f"SchemaError: {e}")

bad_align = {"type": "object", "properties": {
    "tags": {"type": "array",
             "x-eval-align": {"match_by": "fuzzy"},
             "items": {"type": "string"}},
}}
try:
    parse_eval_schema(annotate_xeval(bad_align))
except SchemaError as e:
    print(f"SchemaError: {e}")

SchemaError: name: Unknown comparator: 'smart_match'
SchemaError: tags: x-eval-align 'match_by' must be one of ['hungarian', 'key_field'], got 'fuzzy'


## Summary: symptom -> likely cause

| Symptom in results | Likely cause | Caught by |
|---|---|---|
| Field always `hallucination` though the value looks right | field missing from schema, or at a different path (Cases 1-2) | `validate_gold()` raises |
| Field never scored at all | field only in the schema, or path mismatch hides it (Cases 2-3) | `validate_gold()` warns |
| Identical values score 0 with `reason=type_error` | wrong declared leaf type (Case 5) | grep results for `type_error` |
| Container scored as one opaque value, no per-element detail | wrong declared container type (Case 6) | warning in logs |
| Array all-omission + all-hallucination on every record | wrong alignment key (Case 7) | parse-time warning in logs |
| `SchemaError` before any scoring | invalid x-eval value (Case 8) | fail-fast |

Takeaways:

1. **Always run `validate_gold(gold, schema)` before evaluating.** It catches
   the missing-field and wrong-path cases -- the ones that most badly distort
   results.
2. It can't catch wrong leaf types or a wrong alignment key -- for those,
   watch the **warnings** and check for `reason=type_error` in field results.
3. If a field that should be easy scores 0 across **every** record, suspect
   the schema before the extractor.